# Data Exploration

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from torchvision.utils import make_grid
from src.data.datamodule import BaseDataModule
from omegaconf import OmegaConf

%matplotlib inline

In [ ]:
config = OmegaConf.create({
    "batch_size": 16,
    "num_workers": 0,
    "seed": 42,
    "pin_memory": False,
    "dataset": {
        "name": "image_classification",
        "data_dir": "./data",
        "image_size": 224,
        "num_classes": 10
    }
})

class DummyDataModule(BaseDataModule):
    def __init__(self, config):
        super().__init__(config)
        self.num_samples = 100
        self.num_classes_val = config.get("dataset", {}).get("num_classes", 10)
        self.img_size = config.get("dataset", {}).get("image_size", 224)

    def setup(self, stage=None):
        from torch.utils.data import TensorDataset
        images = torch.randn(self.num_samples, 3, self.img_size, self.img_size)
        labels = torch.randint(0, self.num_classes_val, (self.num_samples,))
        self._train_dataset = TensorDataset(images, labels)
        val_images = torch.randn(self.num_samples // 2, 3, self.img_size, self.img_size)
        val_labels = torch.randint(0, self.num_classes_val, (self.num_samples // 2,))
        self._val_dataset = TensorDataset(val_images, val_labels)

    @property
    def num_classes(self):
        return self.num_classes_val

    def _create_dataloader(self, dataset, shuffle=False, drop_last=False):
        from torch.utils.data import DataLoader
        return DataLoader(dataset, batch_size=self.batch_size, shuffle=shuffle, num_workers=self.num_workers, drop_last=drop_last)

dm = DummyDataModule(config)
dm.setup()
print(f"Train samples: {len(dm.train_dataset)}")
print(f"Val samples: {len(dm.val_dataset)}")

In [ ]:
dataloader = dm.train_dataloader()
sample_batch = next(iter(dataloader))
images, labels = sample_batch

images = (images - images.min()) / (images.max() - images.min() + 1e-8)

n_show = min(16, images.size(0))
grid = make_grid(images[:n_show], nrow=4, normalize=False)
grid = grid.permute(1, 2, 0).numpy()

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(grid)
titles = [f"Class {labels[i].item()}" for i in range(n_show)]
cols = 4
rows = (n_show + cols - 1) // cols
for idx, title in enumerate(titles):
    r, c = divmod(idx, cols)
    x = c / cols + 0.5 / cols
    y = 1.0 - (r + 0.92) / rows
    ax.text(x, y, title, transform=ax.transAxes, fontsize=8, ha='center', va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
ax.axis('off')
ax.set_title('Sample Training Batch')
plt.tight_layout()
plt.show()

In [ ]:
all_labels = []
for batch in dm.train_dataloader():
    _, labels = batch
    all_labels.extend(labels.tolist())

label_counts = pd.Series(all_labels).value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(len(label_counts)), label_counts.values, color=sns.color_palette("husl", len(label_counts)))
ax.set_xlabel("Class")
ax.set_ylabel("Count")
ax.set_title("Class Distribution (Training Set)")
ax.set_xticks(range(len(label_counts)))
ax.set_xticklabels([f"Class {i}" for i in label_counts.index], rotation=45)
for bar, count in zip(bars, label_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5, str(count),
            ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
train_size = len(dm.train_dataset)
val_size = len(dm.val_dataset)
img_dim = dm.img_size
n_classes = dm.num_classes_val

all_train_labels = []
for batch in dm.train_dataloader():
    _, labels = batch
    all_train_labels.extend(labels.tolist())

all_val_labels = []
for batch in dm.val_dataloader():
    _, labels = batch
    all_val_labels.extend(labels.tolist())

print("=" * 50)
print("DATASET SUMMARY")
print("=" * 50)
print(f"Image dimensions:       {img_dim} x {img_dim} x 3")
print(f"Number of classes:      {n_classes}")
print(f"Train set size:         {train_size}")
print(f"Val set size:           {val_size}")
print(f"Train/Val ratio:        {train_size / max(val_size, 1):.1f}")
print(f"Unique train classes:   {len(set(all_train_labels))}")
print(f"Unique val classes:     {len(set(all_val_labels))}")
print(f"Min samples per class:  {min(pd.Series(all_train_labels).value_counts())}")
print(f"Max samples per class:  {max(pd.Series(all_train_labels).value_counts())}")
print(f"Mean samples per class: {np.mean(pd.Series(all_train_labels).value_counts()):.1f}")
print("=" * 50)

## Transform Testing

In [ ]:
from PIL import Image
import torchvision.transforms.functional as TF

sample_img = (torch.rand(3, 256, 256) * 255).byte()
sample_pil = TF.to_pil_image(sample_img)

from torchvision import transforms as T

train_transform = T.Compose([
    T.RandomResizedCrop(224),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.3, contrast=0.3),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def unnormalize(tensor, mean, std):
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    return tensor * std + mean

fig, axes = plt.subplots(2, 5, figsize=(14, 5))
for i in range(5):
    t_img = unnormalize(train_transform(sample_pil), [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    t_img = torch.clamp(t_img.permute(1, 2, 0), 0, 1)
    axes[0, i].imshow(t_img)
    axes[0, i].set_title(f"Train {i+1}")
    axes[0, i].axis('off')

    v_img = unnormalize(val_transform(sample_pil), [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    v_img = torch.clamp(v_img.permute(1, 2, 0), 0, 1)
    axes[1, i].imshow(v_img)
    axes[1, i].set_title(f"Val {i+1}")
    axes[1, i].axis('off')

axes[0, 0].set_ylabel("Train")
axes[1, 0].set_ylabel("Val")
plt.suptitle("Train vs Validation Transforms", fontsize=14)
plt.tight_layout()
plt.show()